# EDA complementario
Ejecutar desde un kernel limpio. Los CSV derivados se guardan en `reports/runs/eda`.
Las comparaciones de interacciones son exploratorias: las variables se eligieron con todo este conjunto;
no constituyen evaluación final ni reemplazan la validación anidada por repeticiones.
El movimiento 014 se excluye por el archivo 014_1.npy dañado. Ver `docs/datos.md`.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "rehab" / "data.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Abre Jupyter desde la raíz del repositorio REHAB.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
from rehab.data import DATA_DIR

import numpy as np
import pandas as pd
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from rehab.data import validate_inputs
validate_inputs()


In [ ]:
ruta = DATA_DIR

# ------------------------------------------------------------
# Canales según el artículo
# ------------------------------------------------------------

canales_1 = [
    "pitch1",
    "yaw1",
    "roll1",
    "pitch2",
    "yaw2",
    "roll2"
]

canales_2 = [
    "f1",
    "f2",
    "f3",
    "f4",
    "f5",
    "pitch3"
]

estadisticas = [
    "mean",
    "std",
    "median",
    "min",
    "max",
    "range",
    "iqr",
    "rms",
    "energy",
    "mad_diff"
]


# ------------------------------------------------------------
# Función para extraer features de una señal 1D
# ------------------------------------------------------------

def extraer_features(x):

    q25 = np.percentile(x, 25)
    q75 = np.percentile(x, 75)

    diffs = np.diff(x)

    return {
        "mean": np.mean(x),
        "std": np.std(x),
        "median": np.median(x),
        "min": np.min(x),
        "max": np.max(x),
        "range": np.max(x) - np.min(x),
        "iqr": q75 - q25,
        "rms": np.sqrt(np.mean(x ** 2)),
        "energy": np.sum(x ** 2),
        "mad_diff": np.mean(np.abs(diffs))
    }


# ------------------------------------------------------------
# Construcción del DataFrame
# ------------------------------------------------------------

filas = []

# Movimientos 000 a 015, ignorando 014
movimientos = [
    f"{i:03d}"
    for i in range(16)
    if i != 14
]

for movimiento in movimientos:

    archivo_1 = ruta / f"{movimiento}_1.npy"
    archivo_2 = ruta / f"{movimiento}_2.npy"

    # --------------------------------------------------------
    # Validar existencia
    # --------------------------------------------------------

    if not archivo_1.exists():
        print(f"No existe: {archivo_1.name}")
        continue

    if not archivo_2.exists():
        print(f"No existe: {archivo_2.name}")
        continue

    # --------------------------------------------------------
    # Cargar ambos sensores
    # --------------------------------------------------------

    try:
        datos_1 = np.load(archivo_1)
        datos_2 = np.load(archivo_2)

    except Exception as e:
        print(f"Error en movimiento {movimiento}: {e}")
        continue

    # Esperamos:
    # datos_1 -> (n_muestras, 880, 6)
    # datos_2 -> (n_muestras, 880, 6)

    if datos_1.ndim != 3 or datos_2.ndim != 3:
        print(
            f"{movimiento}: dimensiones inesperadas "
            f"{datos_1.shape}, {datos_2.shape}"
        )
        continue

    # --------------------------------------------------------
    # Verificar que correspondan las mismas muestras
    # --------------------------------------------------------

    if datos_1.shape[0] != datos_2.shape[0]:
        print(
            f"ERROR {movimiento}: diferente número de muestras "
            f"_1={datos_1.shape[0]}, _2={datos_2.shape[0]}"
        )
        continue

    if datos_1.shape[1] != datos_2.shape[1]:
        print(
            f"ERROR {movimiento}: diferente número de tiempos "
            f"_1={datos_1.shape[1]}, _2={datos_2.shape[1]}"
        )
        continue

    if datos_1.shape[2] != 6 or datos_2.shape[2] != 6:
        print(
            f"ERROR {movimiento}: se esperaban 6 canales por archivo"
        )
        continue

    n_muestras = datos_1.shape[0]

    print(
        f"{movimiento}: {n_muestras} muestras "
        f"| _1 {datos_1.shape} "
        f"| _2 {datos_2.shape}"
    )

    # --------------------------------------------------------
    # Cada índice rep corresponde a la MISMA muestra
    # en _1 y _2
    # --------------------------------------------------------

    for rep in range(n_muestras):

        muestra_1 = datos_1[rep]  # (880, 6)
        muestra_2 = datos_2[rep]  # (880, 6)

        fila = {
            "movimiento": movimiento,
        }

        # ----------------------------------------------------
        # Features de *_1.npy
        # ----------------------------------------------------

        for i, canal in enumerate(canales_1):

            x = muestra_1[:, i]

            features = extraer_features(x)

            for nombre_stat, valor in features.items():
                fila[f"{canal}_{nombre_stat}"] = valor

        # ----------------------------------------------------
        # Features de *_2.npy
        # ----------------------------------------------------

        for i, canal in enumerate(canales_2):

            x = muestra_2[:, i]

            features = extraer_features(x)

            for nombre_stat, valor in features.items():
                fila[f"{canal}_{nombre_stat}"] = valor

        filas.append(fila)


# ------------------------------------------------------------
# DataFrame final
# ------------------------------------------------------------

df_eda = pd.DataFrame(filas)

df_eda["movimiento"] = (
    df_eda["movimiento"]
    .astype("category")
)

print("\n" + "=" * 60)
print("RESULTADO FINAL")
print("=" * 60)

print("Shape:", df_eda.shape)

print(
    "Número de muestras:",
    len(df_eda)
)

print(
    "Número de movimientos:",
    df_eda["movimiento"].nunique()
)

print(
    "Número de features numéricas:",
    df_eda.select_dtypes(
        include=np.number
    ).shape[1]  # quitamos repeticion
)

display(df_eda.head())

In [ ]:
# Perfil descriptivo sin una dependencia externa no declarada.
REPORTS = ROOT / "reports" / "runs" / "eda"
REPORTS.mkdir(parents=True, exist_ok=True)
(REPORTS / "perfil_descriptivo.html").write_text(
    df_eda.describe(include="all").to_html(), encoding="utf-8"
)
print(REPORTS / "perfil_descriptivo.html")

In [ ]:
df_eda.to_csv(REPORTS / "df_eda.csv", index=False)

In [ ]:
df_eda.describe(include="all")

In [ ]:
canales = canales_1 + canales_2

comprobacion = []

for canal in canales:
    mean_col = f"{canal}_mean"
    std_col = f"{canal}_std"
    min_col = f"{canal}_min"
    max_col = f"{canal}_max"
    range_col = f"{canal}_range"
    rms_col = f"{canal}_rms"
    energy_col = f"{canal}_energy"

    comprobacion.append({
        "canal": canal,

        "max_abs_mean":
            df_eda[mean_col].abs().max(),

        "max_diferencia_std_rms":
            (df_eda[std_col] - df_eda[rms_col])
            .abs()
            .max(),

        "max_error_energy":
            (
                df_eda[energy_col]
                - 880 * df_eda[rms_col] ** 2
            )
            .abs()
            .max(),

        "max_error_range":
            (
                df_eda[range_col]
                - (
                    df_eda[max_col]
                    - df_eda[min_col]
                )
            )
            .abs()
            .max()
    })

comprobacion = pd.DataFrame(comprobacion)

display(comprobacion)

In [ ]:
feature_cols = df_eda.select_dtypes(
    include=np.number
).columns.tolist()

diagnostico_columnas = pd.DataFrame({
    "caracteristica": feature_cols,
    "n_unicos": [
        df_eda[col].nunique(dropna=True)
        for col in feature_cols
    ],
    "std": [
        df_eda[col].std()
        for col in feature_cols
    ],
    "min": [
        df_eda[col].min()
        for col in feature_cols
    ],
    "max": [
        df_eda[col].max()
        for col in feature_cols
    ],
    "faltantes": [
        df_eda[col].isna().sum()
        for col in feature_cols
    ]
})

diagnostico_columnas["rango"] = (
    diagnostico_columnas["max"]
    - diagnostico_columnas["min"]
)

display(
    diagnostico_columnas
    .sort_values(["n_unicos", "std"])
    .head(30)
)

In [ ]:
columnas_constantes = diagnostico_columnas.loc[
    diagnostico_columnas["n_unicos"] <= 1,
    "caracteristica"
].tolist()

columnas_casi_constantes = diagnostico_columnas.loc[
    diagnostico_columnas["std"] < 1e-10,
    "caracteristica"
].tolist()

print("Constantes:")
print(columnas_constantes)

print("\nPrácticamente constantes:")
print(columnas_casi_constantes)

In [ ]:
muestras_sin_variacion = []

for canal in canales:

    col_range = f"{canal}_range"

    n_ceros = np.isclose(
        df_eda[col_range],
        0,
        atol=1e-12
    ).sum()

    muestras_sin_variacion.append({
        "canal": canal,
        "muestras_range_cero": n_ceros,
        "porcentaje": (
            100 * n_ceros / len(df_eda)
        )
    })

muestras_sin_variacion = pd.DataFrame(
    muestras_sin_variacion
).sort_values(
    "porcentaje",
    ascending=False
)

display(muestras_sin_variacion)

In [ ]:
range_cero_por_movimiento = []

for canal in canales:

    temporal = df_eda[
        ["movimiento", f"{canal}_range"]
    ].copy()

    temporal["range_cero"] = np.isclose(
        temporal[f"{canal}_range"],
        0,
        atol=1e-12
    )

    resumen = (
        temporal
        .groupby(
            "movimiento",
            observed=True
        )["range_cero"]
        .agg(["sum", "mean"])
        .reset_index()
    )

    resumen["canal"] = canal
    resumen["porcentaje"] = (
        100 * resumen["mean"]
    )

    range_cero_por_movimiento.append(
        resumen[
            [
                "movimiento",
                "canal",
                "sum",
                "porcentaje"
            ]
        ]
    )

range_cero_por_movimiento = pd.concat(
    range_cero_por_movimiento,
    ignore_index=True
)

display(
    range_cero_por_movimiento.sort_values(
        "porcentaje",
        ascending=False
    ).head(40)
)

In [ ]:
muestras_nulas_por_canal = []

for canal in canales:

    mascara = (
        np.isclose(
            df_eda[f"{canal}_std"],
            0,
            atol=1e-12
        )
        &
        np.isclose(
            df_eda[f"{canal}_min"],
            0,
            atol=1e-12
        )
        &
        np.isclose(
            df_eda[f"{canal}_max"],
            0,
            atol=1e-12
        )
    )

    muestras_nulas_por_canal.append({
        "canal": canal,
        "muestras_nulas": mascara.sum(),
        "porcentaje": (
            100 * mascara.mean()
        )
    })

muestras_nulas_por_canal = pd.DataFrame(
    muestras_nulas_por_canal
).sort_values(
    "porcentaje",
    ascending=False
)

display(muestras_nulas_por_canal)

In [ ]:
canales = canales_1 + canales_2

estadisticas_conservar = [
    "std",
    "median",
    "min",
    "max",
    "iqr",
    "mad_diff"
]

estadisticas_redundantes = [
    "mean",
    "range",
    "rms",
    "energy"
]

features_reducidas = [
    f"{canal}_{stat}"
    for canal in canales
    for stat in estadisticas_conservar
]

df_eda_reducido = df_eda[
    ["movimiento"] + features_reducidas
].copy()

print("DataFrame original:", df_eda.shape)
print("DataFrame reducido:", df_eda_reducido.shape)

print(
    "Características numéricas conservadas:",
    len(features_reducidas)
)

display(df_eda_reducido.head())

In [ ]:
df_eda_reducido.to_csv(REPORTS / "df_eda_reducido.csv", index=False)

In [ ]:
resumen_univariado = (
    df_eda_reducido[features_reducidas]
    .describe()
    .T
)

resumen_univariado["rango_total"] = (
    resumen_univariado["max"]
    - resumen_univariado["min"]
)

resumen_univariado["iqr_describe"] = (
    resumen_univariado["75%"]
    - resumen_univariado["25%"]
)

resumen_univariado["coeficiente_variacion"] = (
    resumen_univariado["std"]
    / resumen_univariado["mean"].abs()
        .replace(0, np.nan)
)

display(resumen_univariado)

In [ ]:
resumen_univariado["asimetria"] = (
    df_eda_reducido[features_reducidas]
    .skew()
)

resumen_univariado["curtosis"] = (
    df_eda_reducido[features_reducidas]
    .kurtosis()
)

resumen_univariado["porcentaje_ceros"] = [
    100 * np.isclose(
        df_eda_reducido[col],
        0,
        atol=1e-12
    ).mean()
    for col in features_reducidas
]

resumen_univariado = (
    resumen_univariado
    .sort_values(
        "asimetria",
        key=lambda x: x.abs(),
        ascending=False
    )
)

display(
    resumen_univariado[
        [
            "mean",
            "std",
            "min",
            "25%",
            "50%",
            "75%",
            "max",
            "asimetria",
            "curtosis",
            "porcentaje_ceros"
        ]
    ].head(25)
)

In [ ]:
def separar_nombre_feature(nombre):

    for stat in sorted(
        estadisticas_conservar,
        key=len,
        reverse=True
    ):
        sufijo = f"_{stat}"

        if nombre.endswith(sufijo):
            canal = nombre[:-len(sufijo)]
            return canal, stat

    return None, None

In [ ]:
nombres_separados = [
    separar_nombre_feature(nombre)
    for nombre in resumen_univariado.index
]

resumen_univariado[
    ["canal", "estadistica"]
] = pd.DataFrame(
    nombres_separados,
    index=resumen_univariado.index
)

display(
    resumen_univariado[
        [
            "canal",
            "estadistica",
            "mean",
            "std",
            "asimetria",
            "curtosis",
            "porcentaje_ceros"
        ]
    ].head()
)

In [ ]:
resumen_por_estadistica = (
    resumen_univariado
    .groupby("estadistica")
    .agg(
        desviacion_promedio=("std", "mean"),
        asimetria_media=("asimetria", "mean"),
        asimetria_maxima=("asimetria", lambda x: x.abs().max()),
        curtosis_media=("curtosis", "mean"),
        porcentaje_ceros_promedio=("porcentaje_ceros", "mean")
    )
    .sort_values(
        "asimetria_maxima",
        ascending=False
    )
)

display(resumen_por_estadistica)

In [ ]:
def graficar_histogramas_estadistica(
    df,
    estadistica,
    bins=35
):
    fig, axes = plt.subplots(
        3,
        4,
        figsize=(18, 12)
    )

    axes = axes.flatten()

    for ax, canal in zip(axes, canales):

        columna = f"{canal}_{estadistica}"

        sns.histplot(
            data=df,
            x=columna,
            bins=bins,
            kde=True,
            ax=ax,
            color="steelblue"
        )

        ax.set_title(columna)
        ax.set_xlabel("")
        ax.set_ylabel("Frecuencia")

    plt.suptitle(
        f"Distribuciones de {estadistica}",
        fontsize=16,
        y=1.01
    )

    plt.tight_layout()
    plt.show()

In [ ]:
graficar_histogramas_estadistica(
    df_eda_reducido,
    "std"
)

In [ ]:
graficar_histogramas_estadistica(
    df_eda_reducido,
    "median"
)

graficar_histogramas_estadistica(
    df_eda_reducido,
    "min"
)

graficar_histogramas_estadistica(
    df_eda_reducido,
    "max"
)

graficar_histogramas_estadistica(
    df_eda_reducido,
    "iqr"
)

graficar_histogramas_estadistica(
    df_eda_reducido,
    "mad_diff"
)

In [ ]:
def graficar_boxplots_estadistica(
    df,
    estadistica
):
    columnas = [
        f"{canal}_{estadistica}"
        for canal in canales
    ]

    datos_long = df[columnas].melt(
        var_name="caracteristica",
        value_name="valor"
    )

    plt.figure(figsize=(16, 6))

    sns.boxplot(
        data=datos_long,
        x="caracteristica",
        y="valor",
        showfliers=False,
        color="lightblue"
    )

    plt.title(
        f"Dispersión global de {estadistica} por canal"
    )
    plt.xlabel("")
    plt.ylabel("Valor")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
graficar_boxplots_estadistica(
    df_eda_reducido,
    "std"
)

graficar_boxplots_estadistica(
    df_eda_reducido,
    "iqr"
)

graficar_boxplots_estadistica(
    df_eda_reducido,
    "mad_diff"
)

In [ ]:
resumen_outliers = []

for feature in features_reducidas:

    serie = df_eda_reducido[
        feature
    ].dropna()

    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    mascara_outlier = (
        (serie < limite_inferior)
        | (serie > limite_superior)
    )

    resumen_outliers.append({
        "caracteristica": feature,
        "limite_inferior": limite_inferior,
        "limite_superior": limite_superior,
        "n_outliers": mascara_outlier.sum(),
        "porcentaje_outliers": (
            100 * mascara_outlier.mean()
        )
    })

resumen_outliers = (
    pd.DataFrame(resumen_outliers)
    .sort_values(
        "porcentaje_outliers",
        ascending=False
    )
)

display(resumen_outliers.head(25))

In [ ]:
corr_spearman = (
    df_eda_reducido[features_reducidas]
    .corr(method="spearman")
)

plt.figure(figsize=(20, 17))

sns.heatmap(
    corr_spearman,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    xticklabels=True,
    yticklabels=True
)

plt.title(
    "Correlaciones de Spearman entre las características"
)

plt.xticks(
    rotation=90,
    fontsize=6
)

plt.yticks(
    rotation=0,
    fontsize=6
)

plt.tight_layout()
plt.show()

In [ ]:
corr_abs = corr_spearman.abs()

mascara_superior = np.triu(
    np.ones(corr_abs.shape),
    k=1
).astype(bool)

pares_correlacionados = (
    corr_abs
    .where(mascara_superior)
    .stack()
    .reset_index()
)

pares_correlacionados.columns = [
    "caracteristica_1",
    "caracteristica_2",
    "correlacion_absoluta"
]

pares_correlacionados = (
    pares_correlacionados[
        pares_correlacionados[
            "correlacion_absoluta"
        ] >= 0.90
    ]
    .sort_values(
        "correlacion_absoluta",
        ascending=False
    )
)

display(pares_correlacionados.head(40))

In [ ]:
medias_movimiento = (
    df_eda_reducido
    .groupby(
        "movimiento",
        observed=True
    )[features_reducidas]
    .mean()
)

media_global = (
    df_eda_reducido[features_reducidas]
    .mean()
)

std_global = (
    df_eda_reducido[features_reducidas]
    .std()
    .replace(0, np.nan)
)

medias_movimiento_z = (
    medias_movimiento
    - media_global
) / std_global

In [ ]:
plt.figure(figsize=(23, 9))

sns.heatmap(
    medias_movimiento_z,
    cmap="coolwarm",
    center=0,
    linewidths=0.15
)

plt.title(
    "Perfil multivariado estandarizado por movimiento"
)
plt.xlabel("Características")
plt.ylabel("Movimiento")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder

X_interacciones = (
    df_eda_reducido[features_reducidas]
    .copy()
)

y_interacciones = (
    df_eda_reducido["movimiento"]
    .astype(str)
)

# Convertimos las etiquetas a valores numéricos
label_encoder = LabelEncoder()

y_codificada = label_encoder.fit_transform(
    y_interacciones
)

informacion_mutua = mutual_info_classif(
    X_interacciones,
    y_codificada,
    random_state=42
)

ranking_mi = (
    pd.DataFrame({
        "caracteristica": features_reducidas,
        "informacion_mutua": informacion_mutua
    })
    .sort_values(
        "informacion_mutua",
        ascending=False
    )
    .reset_index(drop=True)
)

display(ranking_mi.head(20))

In [ ]:
plt.figure(figsize=(10, 8))

sns.barplot(
    data=ranking_mi.head(20),
    x="informacion_mutua",
    y="caracteristica",
    color="darkorange"
)

plt.title(
    "Características con mayor información sobre el movimiento"
)
plt.xlabel("Información mutua")
plt.ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
corr_interacciones = (
    df_eda_reducido[features_reducidas]
    .corr(method="spearman")
    .abs()
)

features_interaccion = []

for feature in ranking_mi["caracteristica"]:

    if len(features_interaccion) == 0:
        features_interaccion.append(feature)

    else:
        correlaciones_previas = [
            corr_interacciones.loc[
                feature,
                seleccionada
            ]
            for seleccionada in features_interaccion
        ]

        if all(
            correlacion < 0.75
            for correlacion in correlaciones_previas
        ):
            features_interaccion.append(feature)

    if len(features_interaccion) == 4:
        break

print(
    "Características seleccionadas:"
)

for feature in features_interaccion:
    mi_feature = ranking_mi.loc[
        ranking_mi["caracteristica"] == feature,
        "informacion_mutua"
    ].iloc[0]

    print(
        f"- {feature}: "
        f"MI = {mi_feature:.4f}"
    )

In [ ]:
display(
    df_eda_reducido[
        features_interaccion
    ]
    .corr(method="spearman")
    .round(3)
)

In [ ]:
columnas_pairplot = features_interaccion + ["movimiento"]
partes = []

for movimiento, grupo in df_eda_reducido.groupby(
    "movimiento",
    observed=True
):
    partes.append(
        grupo[columnas_pairplot].sample(
            n=min(120, len(grupo)),
            random_state=42
        )
    )

muestra_pairplot = pd.concat(
    partes,
    ignore_index=True
)

In [ ]:
g = sns.pairplot(
    data=muestra_pairplot,
    vars=features_interaccion,
    hue="movimiento",
    palette="tab20",
    corner=True,
    diag_kind="hist",
    plot_kws={
        "alpha": 0.55,
        "s": 22
    },
    diag_kws={
        "alpha": 0.45
    }
)

g.fig.suptitle(
    "Interacciones entre características informativas",
    y=1.02,
    fontsize=16
)

plt.show()

In [ ]:
feature_x = features_interaccion[0]
feature_y = features_interaccion[1]

In [ ]:
centroides = (
    df_eda_reducido
    .groupby(
        "movimiento",
        observed=True
    )[
        [feature_x, feature_y]
    ]
    .mean()
    .reset_index()
)

In [ ]:
plt.figure(figsize=(12, 8))

sns.scatterplot(
    data=df_eda_reducido,
    x=feature_x,
    y=feature_y,
    hue="movimiento",
    palette="tab20",
    alpha=0.15,
    s=20,
    legend=False
)

sns.scatterplot(
    data=centroides,
    x=feature_x,
    y=feature_y,
    hue="movimiento",
    palette="tab20",
    s=180,
    edgecolor="black",
    linewidth=1.2
)

for _, fila in centroides.iterrows():
    plt.text(
        fila[feature_x],
        fila[feature_y],
        str(fila["movimiento"]),
        fontsize=9,
        fontweight="bold",
        ha="center",
        va="center"
    )

plt.title(
    f"Interacción entre {feature_x} y {feature_y}"
)
plt.legend(
    title="Movimiento",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)
plt.tight_layout()
plt.show()

In [ ]:
from itertools import combinations

pares_features = list(
    combinations(
        features_interaccion,
        2
    )
)

fig, axes = plt.subplots(
    2,
    3,
    figsize=(20, 12)
)

axes = axes.flatten()

for ax, (feature_x, feature_y) in zip(
    axes,
    pares_features
):
    sns.scatterplot(
        data=muestra_pairplot,
        x=feature_x,
        y=feature_y,
        hue="movimiento",
        palette="tab20",
        alpha=0.55,
        s=20,
        legend=False,
        ax=ax
    )

    ax.set_title(
        f"{feature_x}\nvs. {feature_y}"
    )

plt.suptitle(
    "Comparación de interacciones entre características",
    fontsize=16
)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

modelo_interaccion = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42
)

In [ ]:
def comparar_interaccion(
    df,
    feature_1,
    feature_2
):
    y = LabelEncoder().fit_transform(df["movimiento"].astype(str))

    resultados = []

    conjuntos = {
        feature_1: [feature_1],
        feature_2: [feature_2],
        "combinacion": [
            feature_1,
            feature_2
        ]
    }

    for nombre, columnas in conjuntos.items():

        scores = cross_val_score(
            modelo_interaccion,
            df[columnas],
            y,
            cv=cv,
            scoring="f1_macro",
            n_jobs=1
        )

        resultados.append({
            "variables": nombre,
            "f1_macro_promedio": scores.mean(),
            "desviacion": scores.std()
        })

    return pd.DataFrame(resultados)

In [ ]:
resultados_interacciones = []

for feature_1, feature_2 in pares_features:

    resultado = comparar_interaccion(
        df_eda_reducido,
        feature_1,
        feature_2
    )

    resultado["feature_1"] = feature_1
    resultado["feature_2"] = feature_2

    resultados_interacciones.append(
        resultado
    )

resultados_interacciones = pd.concat(
    resultados_interacciones,
    ignore_index=True
)

display(resultados_interacciones)